In [1]:
# libraries for data loading and system/path settings
import json
import sys
import warnings
import os
from pathlib import Path

# libraries for model building and training
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, logging
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.classification import train_bert
from utils.evaluation import run_testset_ner, evaluate_seqeval, mention_level_evaluation, sentence_level_evaluation

# global settings to suppress unproblematic warning messages
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="The sentencepiece tokenizer")

In [2]:
# load the annotated data in json format
with open("../../../01_data/annotations/annotations_gpt_augmentations_corrected.json", "r") as f:
    data = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

In [3]:
def tokenization_labelling(text, entities, tokenizer, tag2id, max_len):

    # get the encoding of the sentence
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True,
                         max_length=max_len, padding="max_length")
    
    # create preliminary list with O tags for all tokens
    tags = ["O"] * len(encoding.offset_mapping)

    # loop over annotations and extract start and end index as well as the given tag
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_tag = ent["tag"][0:2]

        # loop over all tokens in the sentence and check for overlap
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):
            # continue if it is a special token
            if token_start == token_end == 0:
                continue
            # check for overlap (overlap checking strictly necessary for deberta)
            if token_end > start and token_start < end:
                # assign B-tag if start is equal or smaller (smaller if deberta)
                if token_start <= start:
                    tags[idx] = f"B-{ent_tag}"
                # otherwise it is an inside token
                else:
                    tags[idx] = f"I-{ent_tag}"

    # extract the word ids
    word_ids = encoding.word_ids()

    # ensure propagate B-tags are propagated to all subwords of the same word (only actually relevant for deberta)
    for idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        # if token has B-tag ensure that all other tokens of the same word get I-tag
        if tags[idx].startswith("B-"):
            for j, wid2 in enumerate(word_ids):
                if wid2 == wid and j != idx:
                    tags[j] = f"I-{ent_tag}"

    # convert tags to IDs, masking special tokens
    tag_ids = [-100 if wid is None else tag2id.get(tag, tag2id["O"])
               for tag, wid in zip(tags, encoding.word_ids())]

    return encoding["input_ids"], encoding["attention_mask"], tag_ids, word_ids

In [4]:
class TokenDataset(Dataset):
    def __init__(self, data, tokenizer, tag2id, max_len):
        self.dataset = []
        self.max_len = max_len

        for task in data:
            # get the sentence and all annotations
            text = task["sentence"]
            spans = task["annotations"]

            # tokenize and get all ids
            input_ids, attention_mask, tag_ids, word_ids = tokenization_labelling(text, spans, tokenizer, tag2id, self.max_len)

            # add everything to the dataset list
            self.dataset.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "tag_ids": torch.tensor(tag_ids, dtype=torch.long),
                "word_ids": word_ids})
  

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def custom_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    tag_ids = torch.stack([item["tag_ids"] for item in batch])
    word_ids = [item["word_ids"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "tag_ids": tag_ids,
        "word_ids": word_ids
    }

In [5]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_names = ["roberta-base"] #, "bert-base-cased", "distilbert-base-cased", "microsoft/deberta-v3-base"]
train_data, val_data = train_test_split(data, test_size=0.2, shuffle=True, random_state=42)
optimal_configs = {
    "roberta-base": {
        "best_epoch": 8,
        "best_params": {
            "lr": 9e-06,
            "batch_size": 16,
            "weight_decay": 0.01
        }
    },
    "bert-base-cased": {
        "best_epoch": 8,
        "best_params": {
            "lr": 9e-06,
            "batch_size": 16,
            "weight_decay": 0.01
        }
    },
    "distilbert-base-cased": {
        "best_epoch": 8,
        "best_params": {
            "lr": 4e-06,
            "batch_size": 16,
            "weight_decay": 0.3
        }
    },
    "microsoft/deberta-v3-base": {
        "best_epoch": 8,
        "best_params": {
            "lr": 4e-05,
            "batch_size": 16,
            "weight_decay": 0.3
        }
    }
}

In [6]:
results = {}

for model_name in model_names:

    # get the hyperparameter configuration
    epochs = optimal_configs[model_name]["best_epoch"]
    lr = optimal_configs[model_name]["best_params"]["lr"]
    batch_size = optimal_configs[model_name]["best_params"]["batch_size"]
    weight_decay = optimal_configs[model_name]["best_params"]["weight_decay"]

    # define the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create tensor dataset and dataloaders
    train_dataset = TokenDataset(train_data, tokenizer, tag_to_id, max_len=128)
    val_dataset = TokenDataset(val_data, tokenizer, tag_to_id, max_len=128)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

    # instantiate the model and optimizer
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
        ).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # train the model
    train_bert(train_dataloader, model, optimizer, epochs, device, which_task="ner")

    evaluation_inputs = {
                "seqeval": {},
                "cross_span": {},
                "sentence_level": {}
                }
    

    all_true, all_pred, _ = run_testset_ner(
                        model=model, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric="seqeval"
                        )
    # loop over all metrics and get the ground truth as well as predicted labels for the validation set
    for metric in evaluation_inputs.keys():
        all_true, all_pred, _ = run_testset_ner(
               model=model, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric=metric
               )
        evaluation_inputs[metric] = {
            "all_true": all_true,
            "all_pred":all_pred
            }

    # apply all evaluation functions and save in the dictionary
    metrics_seqeval = evaluate_seqeval(
           evaluation_inputs["seqeval"]["all_true"],
           evaluation_inputs["seqeval"]["all_pred"]
           )
    metrics_cross_span = mention_level_evaluation(
           evaluation_inputs["cross_span"]["all_true"],
            evaluation_inputs["cross_span"]["all_pred"]
            )
    metrics_sentence_level = sentence_level_evaluation(
            evaluation_inputs["sentence_level"]["all_true"],
            evaluation_inputs["sentence_level"]["all_pred"]
            )
    
    results[model_name] = {
        "seqeval": metrics_seqeval,
        "cross-span": metrics_cross_span,
        "sentence_level": metrics_sentence_level
    }

Epoch 1/8


Training: 100%|██████████| 250/250 [01:42<00:00,  2.43it/s, loss=0.0384]


Average training loss: 0.1412
Epoch 2/8


Training: 100%|██████████| 250/250 [01:43<00:00,  2.41it/s, loss=0.0302] 


Average training loss: 0.0517
Epoch 3/8


...

Average training loss: 0.0114
Epoch 7/8


Training: 100%|██████████| 250/250 [01:43<00:00,  2.41it/s, loss=0.0484]  


Average training loss: 0.0103
Epoch 8/8


Training: 100%|██████████| 250/250 [01:43<00:00,  2.42it/s, loss=0.00156] 


Average training loss: 0.0068


In [26]:
# export the metrics
with open("../eval_results/evaluation_metrics_bert.json", "w") as f:
    json.dump(results, f)